<a href="https://colab.research.google.com/github/bhardwajutkarsh313-dotcom/PBEL_3.0/blob/main/fake_news_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile app.py
"""
AI News Lab - Fake News Generator & Detector
=============================================
Educational demo combining GPT-2 (text generation) with a RoBERTa model
fine-tuned specifically for fake-vs-real news classification.

Run:
  pip install -r requirements.txt
  python app.py
"""

import torch
import gradio as gr
from transformers import (
  GPT2Tokenizer,
  GPT2LMHeadModel,
  AutoTokenizer,
  AutoModelForSequenceClassification,
)

GPT2_MODEL_NAME = "gpt2" # swap to "gpt2-medium" for noticeably better (slower) generations
DETECTOR_MODEL_NAME = "winterForestStump/Roberta-fake-news-detector" # fine-tuned for fake/real news, not a blank head

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Loading models on {DEVICE} ...")

gpt2_tokenizer = GPT2Tokenizer.from_pretrained(GPT2_MODEL_NAME)
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token
gpt2_model = GPT2LMHeadModel.from_pretrained(GPT2_MODEL_NAME).to(DEVICE)
gpt2_model.eval()

detector_tokenizer = AutoTokenizer.from_pretrained(DETECTOR_MODEL_NAME)
detector_model = AutoModelForSequenceClassification.from_pretrained(DETECTOR_MODEL_NAME).to(DEVICE)
detector_model.eval()

print("Models loaded.")


def generate_fake_news(prompt, max_new_tokens=150, temperature=0.8, top_p=0.95, top_k=50):
  prompt = (prompt or "").strip()
  if not prompt:
    return "Please enter a headline or opening sentence first."

  try:
    encoded = gpt2_tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
      output_ids = gpt2_model.generate(
        **encoded,
        max_new_tokens=int(max_new_tokens),
        do_sample=True,
        temperature=float(temperature),
        top_p=float(top_p),
        top_k=int(top_k),
        no_repeat_ngram_size=3,
        repetition_penalty=1.2,
        pad_token_id=gpt2_tokenizer.eos_token_id,
      )
    return gpt2_tokenizer.decode(output_ids[0], skip_special_tokens=True)
  except Exception as e:
    print(f"Generation error: {e}")
    return f"Generation failed: {e}"


def detect_news(text):
  text = (text or "").strip()
  if not text:
    return {"Enter some text first": 1.0}

  try:
    inputs = detector_tokenizer(
      text, return_tensors="pt", truncation=True, padding=True, max_length=512
    ).to(DEVICE)
    with torch.no_grad():
      logits = detector_model(**inputs).logits
    probs = torch.softmax(logits, dim=1)[0]

    id2label = detector_model.config.id2label
    return {id2label[i]: float(probs[i]) for i in range(len(probs))}
  except Exception as e:
    print(f"Detection error: {e}")
    return {"Error - check console": 1.0}


theme = gr.themes.Soft(
  primary_hue="violet",
  secondary_hue="slate",
  font=[gr.themes.GoogleFont("Inter"), "sans-serif"],
)

GENERATE_EXAMPLES = [
  "Scientists in Antarctica report a strange metallic sound coming from beneath the ice.",
  "Local city council votes to replace all traffic lights with musical chimes.",
  "A new species of fish was discovered that can breathe air for up to a week.",
]

DETECT_EXAMPLES = [
  "NASA confirmed today that its Perseverance rover recorded new wind sounds on Mars.",
  "Doctors say drinking a spoon of vinegar every hour cures the common cold within a day.",
]

with gr.Blocks(title="AI News Lab") as demo:
  gr.Markdown(
    """
    # AI News Lab
    ### Fake News Generator & Detector - GPT-2 + RoBERTa
    A small educational demo showing how AI can both *write* convincing-sounding text
    and *flag* it. Not a real fact-checking tool - see the note at the bottom.
    """
  )

  with gr.Tab("Generate"):
    with gr.Row():
      with gr.Column(scale=3):
        gen_input = gr.Textbox(
          label="Headline or opening line",
          placeholder="e.g. A mysterious object was spotted in the sky over...",
          lines=2,
        )
        with gr.Accordion("Generation settings", open=False):
          max_new_tokens = gr.Slider(30, 300, value=150, step=10, label="Length (new tokens)")
          temperature = gr.Slider(0.3, 1.5, value=0.8, step=0.05, label="Temperature (creativity)")
          top_p = gr.Slider(0.5, 1.0, value=0.95, step=0.01, label="Top-p")
          top_k = gr.Slider(0, 100, value=50, step=5, label="Top-k")
        generate_btn = gr.Button("Generate", variant="primary")
        gr.Examples(GENERATE_EXAMPLES, inputs=gen_input, label="Try an example")
      with gr.Column(scale=3):
        gen_output = gr.Textbox(label="Generated article", lines=10)

    generate_btn.click(
      generate_fake_news,
      inputs=[gen_input, max_new_tokens, temperature, top_p, top_k],
      outputs=gen_output,
    )

  with gr.Tab("Detect"):
    with gr.Row():
      with gr.Column(scale=3):
        detect_input = gr.Textbox(
          label="News article or statement",
          placeholder="Paste a paragraph to check...",
          lines=8,
        )
        detect_btn = gr.Button("Analyze", variant="primary")
        gr.Examples(DETECT_EXAMPLES, inputs=detect_input, label="Try an example")
      with gr.Column(scale=2):
        detect_output = gr.Label(label="Result", num_top_classes=2)

    detect_btn.click(detect_news, inputs=detect_input, outputs=detect_output)

  gr.Markdown(
    """
    ---
    WARNING: **About accuracy:** the detector uses a RoBERTa model fine-tuned specifically for
    fake/real news classification (not a generic, untrained head), so its confidence
    scores actually reflect learned patterns. It's still a demo model trained on a
    limited dataset, so treat results as illustrative, not a verdict.
    Generated articles are AI-written fiction and shouldn't be shared as real news.
    """
  )

demo.queue()

if __name__ == "__main__":
  demo.launch(theme=theme, share=True)

Writing app.py


In [2]:
!pip install -q transformers torch gradio

In [ ]:
!python app.py